<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    06 · Curación Molecular
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:640px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 3 — El módulo más diferenciador del curso</em>
  </p>
</div>


---
## ¿Por qué la curación de datos es el paso más crítico?

> *"Garbage in, garbage out"* — cualquier modelo entrenado con datos mal curados aprenderá relaciones falsas.

Los datos de ChEMBL vienen de **miles de artículos distintos**, con protocolos diferentes, unidades distintas
y calidades variables. Antes de modelar, necesitamos un dataset **limpio, consistente y confiable**.

### ¿Qué haremos en este notebook?

| Paso | Problema | Solución |
|------|----------|----------|
| 1 | Instalación y carga del dataset raw | — |
| 2 | Diagnóstico: ¿qué tan sucios están los datos? | Inspección visual y estadística |
| 3 | Conversión y homogeneización de unidades | nM, µM, mg/mL → nM único |
| 4 | Validación y lectura de SMILES | `read_smiles()` |
| 5 | Eliminación de sales y fragmentos no deseados | `select_largest_organic_component()` |
| 6 | Estandarización con ChEMBL Structure Pipeline | `chembl_standardizer()` |
| 7 | Pipeline completo: `process_smiles()` | Combina pasos 4–6 |
| 8 | Curación de la actividad biológica | Duplicados, outliers, pActividad |
| 9 | Filtros adicionales de drug-likeness | Lipinski, PAINS, complejidad |
| 10 | Clasificación activo / inactivo | Umbral justificado |
| 11 | Reporte final de curación | Cuánto perdimos y por qué |
| 12 | Guardar dataset curado | CSV listo para Semana 4 |


---
## 1. Instalación y carga del dataset

In [ ]:
# ── Instalar librerías ──────────────────────────────────────────────────────
!pip install chembl_webresource_client chembl-structure-pipeline rdkit --quiet

print("✅ Librerías instaladas")


In [1]:
# ── Importaciones ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from math import log
import os, re, warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, Draw
from rdkit.Chem import FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams
from chembl_webresource_client.new_client import new_client
from chembl_structure_pipeline import standardize_mol

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 55)

print("✅ Todo listo para curar datos moleculares")


✅ Todo listo para curar datos moleculares


[16:25:33] Initializing Normalizer


In [2]:
# ── Opción A: cargar el CSV guardado en NB-DATA-01 ─────────────────────────
# Si ya tienes el archivo del NB-DATA-01, descomenta y ajusta la ruta:
# df_raw = pd.read_csv('dataset_egfr_raw.csv')

# ── Opción B: descargar directamente desde ChEMBL (EGFR como ejemplo) ───────
# Usamos la función chembl_mols del notebook de docking — aquí integrada:

from chembl_webresource_client.new_client import new_client

def chembl_mols(chembl_id):
    """
    Descarga moléculas activas de un target ChEMBL.

    Filtros aplicados en orden (con reporte de cuántas moléculas elimina cada uno):
      n0 → descarga inicial en nM
      n1 → tipo de actividad: IC50, Ki, EC50, Kd
      n2 → confidence score del assay: solo 8 y 9
      n3 → standard_value > 0
      n4 → pActividad >= 5 (IC50 <= 10 µM)
      n5 → deduplicación por SMILES (mayor actividad)
      n6 → peso molecular: 180–900 Da

    Confidence score en ChEMBL:
      9 = medida directa sobre el target puro (gold standard)
      8 = target bien definido con ligera ambigüedad
      < 8 = target inferido o poco confiable → se descartan

    Parámetros
    ----------
    chembl_id : str — ID del target en ChEMBL (ej. 'CHEMBL203')

    Retorna
    -------
    df          : DataFrame con actividades curadas
    target_name : str
    organism    : str
    """
    activity = new_client.activity
    target   = new_client.target
    assay_cl = new_client.assay

    # ── Info del target ───────────────────────────────────────────────────────
    targetInfo  = target.filter(
        target_chembl_id=chembl_id
    ).only('pref_name', 'target_type', 'organism')
    targetData  = pd.DataFrame(targetInfo)
    target_name = targetData['pref_name'][0]
    organism    = targetData['organism'][0]

    print(f"Target: {target_name} ({organism})")
    print("=" * 55)

    # ── n0: Descargar actividades en nM ──────────────────────────────────────
    print("  Descargando actividades desde ChEMBL...")
    listOfActivities = activity.filter(
        target_chembl_id=chembl_id
    ).filter(
        standard_units='nM'
    ).only(
        'canonical_smiles', 'molecule_chembl_id', 'pchembl_value',
        'standard_units', 'standard_value', 'standard_type',
        'assay_chembl_id'          # necesario para cruzar con confidence score
    )

    if not listOfActivities:
        print("❌ ChEMBL ID no encontrado")
        return None, None, None

    df = pd.DataFrame(listOfActivities)
    n0 = len(df)
    print(f"  n0 → descarga inicial (en nM):          {n0:>6}")

    # ── n1: Tipo de actividad ─────────────────────────────────────────────────
    df = df[df['standard_type'].isin(['IC50', 'Ki', 'EC50', 'Kd'])]
    n1 = len(df)
    print(f"  n1 → tipo IC50/Ki/EC50/Kd:              {n1:>6}  (-{n0-n1})")

    # ── n2: Confidence score — descargar por lotes y cruzar ──────────────────
    print("  Obteniendo confidence scores de los assays (por lotes)...")
    assay_ids = df['assay_chembl_id'].dropna().unique().tolist()

    # Lotes de 100 para no generar URLs demasiado largas
    LOTE = 100
    assay_scores = {}
    for i in range(0, len(assay_ids), LOTE):
        ids_lote = assay_ids[i : i + LOTE]
        assays_info = assay_cl.filter(
            assay_chembl_id__in=ids_lote
        ).only('assay_chembl_id', 'confidence_score')
        for a in assays_info:
            assay_scores[a['assay_chembl_id']] = a.get('confidence_score')

    df['confidence_score'] = pd.to_numeric(
        df['assay_chembl_id'].map(assay_scores), errors='coerce'
    )

    # Mostrar distribución completa antes de filtrar
    print(f"  Distribución de confidence scores:")
    dist = df['confidence_score'].value_counts().sort_index(ascending=False)
    for score, count in dist.items():
        marcador = " ← retenidos" if score in [8, 9] else " ← descartados"
        score_str = str(int(score)) if pd.notna(score) else "N/D"
        print(f"    Score {score_str:>2}: {count:>5}{marcador}")

    df = df[df['confidence_score'].isin([8, 9])].copy()
    n2 = len(df)
    print(f"  n2 → confidence score 8 o 9:            {n2:>6}  (-{n1-n2})")

    # ── n3: Valor positivo ────────────────────────────────────────────────────
    df = df.astype({'standard_value': float})
    df = df[df['standard_value'] > 0]
    n3 = len(df)
    print(f"  n3 → standard_value > 0:                {n3:>6}  (-{n2-n3})")

    # ── n4: pActividad ────────────────────────────────────────────────────────
    df['pValue'] = [-log(i / 1e9, 10) for i in df['standard_value']]
    df = df[df['pValue'] >= 5]
    n4 = len(df)
    print(f"  n4 → pActividad >= 5 (IC50 <= 10 µM):  {n4:>6}  (-{n3-n4})")

    # ── n5: Deduplicar por SMILES ─────────────────────────────────────────────
    df = df.sort_values('pValue', ascending=False)
    df = df.drop_duplicates(subset=['canonical_smiles'], keep='first').reset_index(drop=True)
    n5 = len(df)
    print(f"  n5 → SMILES únicos (mayor actividad):   {n5:>6}  (-{n4-n5})")

    # ── n6: Peso molecular ────────────────────────────────────────────────────
    mw = []
    for smi in df['canonical_smiles']:
        try:
            mol = Chem.MolFromSmiles(smi)
            mw.append(Descriptors.MolWt(mol) if mol else 0)
        except:
            mw.append(0)
    df['mol_weight'] = mw
    df = df[(df['mol_weight'] >= 180) & (df['mol_weight'] <= 900)].reset_index(drop=True)
    n6 = len(df)
    print(f"  n6 → peso molecular 180–900 Da:         {n6:>6}  (-{n5-n6})")

    # ── Resumen ───────────────────────────────────────────────────────────────
    print()
    print(f"  ✅ DATASET FINAL: {n6} moléculas ({n6/n0*100:.1f}% del total descargado)")
    print(f"     Eliminadas en total: {n0-n6} de {n0}")

    return df, target_name, organism

# ── Descargar EGFR ───────────────────────────────────────────────────────────
print("Descargando datos de EGFR (CHEMBL203) desde ChEMBL...")
print("Esto puede tardar 1–2 minutos...")
print()
df_raw, TARGET_NAME, TARGET_ORG = chembl_mols('CHEMBL203')
print()
df_raw.head()


Descargando datos de EGFR (CHEMBL203) desde ChEMBL...
Esto puede tardar 1–2 minutos...

Target: Epidermal growth factor receptor (Homo sapiens)
  Descargando actividades desde ChEMBL...
  n0 → descarga inicial (en nM):           28219
  n1 → tipo IC50/Ki/EC50/Kd:               27886  (-333)
  Obteniendo confidence scores de los assays (por lotes)...
  Distribución de confidence scores:
    Score  9: 19974 ← retenidos
    Score  8:  7912 ← retenidos
  n2 → confidence score 8 o 9:             27886  (-0)
  n3 → standard_value > 0:                 27731  (-155)
  n4 → pActividad >= 5 (IC50 <= 10 µM):   24894  (-2837)
  n5 → SMILES únicos (mayor actividad):    12681  (-12213)
  n6 → peso molecular 180–900 Da:          12568  (-113)

  ✅ DATASET FINAL: 12568 moléculas (44.5% del total descargado)
     Eliminadas en total: 15651 de 28219



,assay_chembl_id,canonical_smiles,molecule_chembl_id,pchembl_value,standard_type,standard_units,standard_value,type,units,value,confidence_score,pValue,mol_weight
0,CHEMBL5031734,C=CC(=O)N1CC[C@H](Oc2nc(-c3n[nH]c(=O)[nH]3)cc3ccccc...,CHEMBL5073622,None,IC50,nM,5.012000e-09,pIC50,None,17.3,9,17.299989,351.366
1,CHEMBL4721173,C=CC(=O)Nc1cc(Nc2nccc(-c3cn(C)c4ccccc34)n2)c(OC)cc1...,CHEMBL3353410,None,IC50,nM,2.000000e-03,IC50,nM,0.002,9,11.698970,499.619
2,CHEMBL683040,Brc1cccc(Nc2ncnc3cc4ccccc4cc23)c1,CHEMBL63786,None,IC50,nM,3.000000e-03,IC50,nM,0.003,9,11.522879,350.219
3,CHEMBL680021,CN(C)c1cc2c(Nc3cccc(Br)c3)ncnc2cn1,CHEMBL53711,None,IC50,nM,6.000000e-03,IC50,nM,0.006,8,11.221849,344.216
4,CHEMBL939337,CCOc1cc2ncnc(Nc3cccc(Br)c3)c2cc1OCC,CHEMBL35820,None,IC50,nM,6.000000e-03,IC50,pM,6.0,8,11.221849,388.265


In [9]:
# Guardar el dataframe
df_raw.to_csv(f'D:\\data_git\\curso_datascience\\files\\egfr_chembl_prepared.csv', index=False)

In [7]:
os.getcwd()

'd:\\data_git\\curso_datascience\\notebooks'

---
## 2. Diagnóstico: ¿qué tan sucios están los datos?

Antes de curar, necesitamos entender **qué problemas tiene el dataset**.
Este análisis guía todas las decisiones de curación posteriores.


In [ ]:
# ── Resumen general del dataset raw ─────────────────────────────────────────
print("DIAGNÓSTICO DEL DATASET RAW")
print("=" * 55)
print(f"  Registros totales:           {len(df_raw)}")
print(f"  Compuestos únicos (ChEMBL):  {df_raw['molecule_chembl_id'].nunique()}")
print(f"  SMILES únicos:               {df_raw['canonical_smiles'].nunique()}")
print()

# Valores faltantes
print("Valores faltantes por columna:")
for col in df_raw.columns:
    n_null = df_raw[col].isna().sum()
    pct = n_null / len(df_raw) * 100
    flag = ' ⚠️' if pct > 5 else ''
    print(f"  {col:<25} {n_null:>5} ({pct:.1f}%){flag}")


In [ ]:
# ── Tipos de actividad en el dataset ────────────────────────────────────────
print("DISTRIBUCIÓN POR TIPO DE ACTIVIDAD")
print("=" * 40)
for tipo, n in df_raw['standard_type'].value_counts().items():
    barra = '█' * min(int(n / len(df_raw) * 40), 40)
    print(f"  {tipo:<8} {n:>5}  {barra}")


In [ ]:
# ── Distribución de pActividad (diagnóstico de calidad) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma de pActividad
ax1 = axes[0]
df_raw['pValue'].hist(ax=ax1, bins=40, color='#4a90d9', edgecolor='white', linewidth=0.4)
ax1.axvline(5, color='#e74c3c', linestyle='--', linewidth=1.5, label='pAct=5 (10µM)')
ax1.axvline(6, color='#f39c12', linestyle='--', linewidth=1.5, label='pAct=6 (1µM)')
ax1.axvline(9, color='#27ae60', linestyle='--', linewidth=1.5, label='pAct=9 (1nM)')
ax1.set_xlabel('pActividad (-log₁₀M)', fontsize=11)
ax1.set_ylabel('Frecuencia', fontsize=11)
ax1.set_title(f'Distribución pActividad — {TARGET_NAME}\n(dataset raw, n={len(df_raw)})', fontsize=11)
ax1.legend(fontsize=9)

# Boxplot por tipo de actividad
ax2 = axes[1]
tipos = df_raw['standard_type'].value_counts().index.tolist()
datos_box = [df_raw[df_raw['standard_type']==t]['pValue'].dropna().values for t in tipos]
bp = ax2.boxplot(datos_box, labels=tipos, patch_artist=True)
colores_box = ['#4a90d9','#e74c3c','#27ae60','#f39c12']
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.axhline(5, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax2.set_ylabel('pActividad', fontsize=11)
ax2.set_title('pActividad por tipo de ensayo', fontsize=11)
ax2.set_xlabel('Tipo de actividad', fontsize=11)

plt.tight_layout()
plt.savefig('diagnostico_raw.png', dpi=120, bbox_inches='tight')
plt.show()

print("💡 Observa la diferencia entre IC50 y Ki — ¿tienen distribuciones similares?")
print("   Si no, mezclarlos sin corrección puede sesgar el modelo.")


In [ ]:
# ── Detectar SMILES problemáticos ────────────────────────────────────────────
problemas = {
    'SMILES vacío':       df_raw['canonical_smiles'].isna().sum(),
    'SMILES duplicado':   df_raw['canonical_smiles'].duplicated().sum(),
    'Contiene punto (.)': df_raw['canonical_smiles'].str.contains(r'\.', na=False).sum(),
    'Contiene metal':     df_raw['canonical_smiles'].str.contains(
                              r'\[Fe|\[Cu|\[Zn|\[Pt|\[Au|\[Pd|\[Ru', na=False).sum(),
    'Muy corto (<10 car)':df_raw['canonical_smiles'].str.len().lt(10).sum(),
}

print("PROBLEMAS DETECTADOS EN SMILES (pre-curación)")
print("=" * 45)
for problema, n in problemas.items():
    flag = ' ⚠️' if n > 0 else ' ✅'
    print(f"  {problema:<30} {n:>5}{flag}")

print()
# Mostrar ejemplos de SMILES con punto (mezclas/sales)
mezclas = df_raw[df_raw['canonical_smiles'].str.contains(r'\.', na=False)]
print(f"Ejemplos de SMILES con sales/fragmentos (n={len(mezclas)}):")
for smi in mezclas['canonical_smiles'].head(3):
    print(f"  {smi}")


---
## 3. Conversión y homogeneización de unidades

En ChEMBL conviven datos con distintas unidades para el mismo tipo de actividad.
**Mezclarlas sin conversión es uno de los errores más comunes** en la literatura.


In [ ]:
# ── Tabla de factores de conversión a nM ────────────────────────────────────
FACTORES_A_NM = {
    'nM':    1,
    'nm':    1,
    'uM':    1e3,
    'µM':    1e3,
    'μM':    1e3,
    'mM':    1e6,
    'M':     1e9,
    'pM':    1e-3,
    'fM':    1e-6,
    'ug/ml': None,   # No convertible directamente (depende de MW)
    'mg/ml': None,   # No convertible directamente
    'mg/L':  None,
    '%':     None,   # Inhibición porcentual — no es IC50
}

print("TABLA DE CONVERSIÓN DE UNIDADES A nM")
print("=" * 45)
print(f"  {'Unidad':<12} {'Factor a nM':<18} {'Convertible'}")
print("  " + "-"*42)
for unidad, factor in FACTORES_A_NM.items():
    conv = "✅ Sí" if factor is not None else "❌ No (depende del MW o escala diferente)"
    factor_str = f"× {factor}" if factor is not None else "—"
    print(f"  {unidad:<12} {factor_str:<18} {conv}")


In [ ]:
# ── Función de conversión robusta ────────────────────────────────────────────
def convertir_a_nm(valor, unidad):
    """
    Convierte un valor de actividad a nM.
    
    Parámetros
    ----------
    valor  : float — valor numérico de la actividad
    unidad : str   — unidad original del valor
    
    Retorna
    -------
    float o None — valor en nM, o None si no es convertible
    """
    factores = {
        'nM': 1, 'nm': 1,
        'uM': 1e3, 'µM': 1e3, 'μM': 1e3, 'um': 1e3,
        'mM': 1e6, 'mm': 1e6,
        'M':  1e9,
        'pM': 1e-3, 'pm': 1e-3,
        'fM': 1e-6, 'fm': 1e-6,
    }
    if pd.isna(valor) or pd.isna(unidad):
        return None
    factor = factores.get(str(unidad).strip())
    if factor is None:
        return None
    try:
        return float(valor) * factor
    except (ValueError, TypeError):
        return None

def calcular_pactividad(valor_nm):
    """
    Calcula pActividad = -log10(valor en Molar).
    Robusto a valores 0 o negativos.
    """
    if valor_nm is None or pd.isna(valor_nm) or valor_nm <= 0:
        return None
    return -log(valor_nm / 1e9, 10)

# ── Aplicar conversión al dataset ───────────────────────────────────────────
# En este dataset ya filtramos por nM en la descarga,
# pero aplicamos la función para ser robustos a cualquier entrada futura

df_raw['valor_nM'] = [
    convertir_a_nm(v, u)
    for v, u in zip(df_raw['standard_value'], df_raw['standard_units'])
]

df_raw['pActividad'] = [calcular_pactividad(v) for v in df_raw['valor_nM']]

n_convertidos = df_raw['valor_nM'].notna().sum()
n_no_convertidos = df_raw['valor_nM'].isna().sum()
print(f"Conversión a nM:")
print(f"  Convertidos correctamente: {n_convertidos}")
print(f"  No convertibles:           {n_no_convertidos}")
print()
print(df_raw[['canonical_smiles','standard_type',
              'standard_value','standard_units',
              'valor_nM','pActividad']].head(8).to_string(index=False))


---
## 4. Validación de SMILES con RDKit

El primer paso estructural: verificar que cada SMILES representa una molécula válida que RDKit puede leer.


In [ ]:
# ── Función de validación (del notebook de docking) ─────────────────────────
def read_smiles(smiles):
    """
    Verifica si el SMILES es válido y retorna el objeto mol de RDKit.

    Parámetros
    ----------
    smiles : str — cadena SMILES a validar

    Retorna
    -------
    mol   : rdkit.Chem.Mol o None
    error : str o None — descripción del error si falla
    """
    if pd.isna(smiles) or str(smiles).strip() == '':
        return None, "SMILES vacío o nulo"
    mol = Chem.MolFromSmiles(str(smiles).strip())
    if mol is None:
        return None, "SMILES inválido — RDKit no puede parsearlo"
    return mol, None

# ── Aplicar al dataset ───────────────────────────────────────────────────────
resultados_lectura = [read_smiles(smi) for smi in df_raw['canonical_smiles']]
df_raw['mol_valido']    = [r[0] is not None for r in resultados_lectura]
df_raw['error_lectura'] = [r[1] for r in resultados_lectura]

n_validos   = df_raw['mol_valido'].sum()
n_invalidos = (~df_raw['mol_valido']).sum()

print("VALIDACIÓN DE SMILES")
print("=" * 40)
print(f"  SMILES válidos:   {n_validos:>5}  ✅")
print(f"  SMILES inválidos: {n_invalidos:>5}  ❌")
print()

if n_invalidos > 0:
    print("Ejemplos de SMILES inválidos:")
    invalidos = df_raw[~df_raw['mol_valido']][['canonical_smiles','error_lectura']]
    print(invalidos.head(5).to_string(index=False))


---
## 5. Eliminación de sales y selección del fragmento orgánico principal

Muchos SMILES en ChEMBL contienen **contraiones y sales** (representados con un punto `.` en el SMILES).
Por ejemplo: `[Na+].[O-]C(=O)c1ccccc1` — el carboxilato de sodio del benzoato.

Necesitamos quedarnos solo con el **fragmento farmacológicamente activo**.


In [ ]:
# ── Función de validación molecular (del notebook de docking) ────────────────
def is_valid(mol):
    """
    Valida que un fragmento sea una molécula orgánica drug-like:
    - Solo contiene elementos orgánicos permitidos
    - Tiene al menos un anillo
    - Tiene al menos un aceptor de H-bond

    Parámetros
    ----------
    mol : rdkit.Chem.Mol

    Retorna
    -------
    bool — True si el fragmento es válido
    """
    organic_elements = {'H','B','C','N','O','F','Si','P','S','Cl','Br','I'}
    for atom in mol.GetAtoms():
        if atom.GetSymbol() not in organic_elements:
            return False
    if Descriptors.RingCount(mol) < 1:
        return False
    if Descriptors.NumHAcceptors(mol) < 1:
        return False
    return True


# ── Función principal: seleccionar el fragmento más grande (del notebook de docking) ──
def select_largest_organic_component(mol):
    """
    Selecciona el componente orgánico más grande y válido de una molécula,
    excluyendo sales comunes y contraiones.

    Criterios de selección:
    - Solo fragmentos con elementos orgánicos
    - Al menos un anillo y un aceptor de H-bond
    - El fragmento más grande debe representar ≥60% del total de átomos pesados
    - Se excluyen sales comunes (tosiato, trifluoroacetato, oxalato, etc.)

    Parámetros
    ----------
    mol : rdkit.Chem.Mol — molécula con posibles sales/fragmentos

    Retorna
    -------
    largest_fragment       : rdkit.Chem.Mol o None
    num_fragments          : int — total de fragmentos encontrados
    num_unique_fragments   : int — fragmentos únicos
    error                  : str o None
    """
    # Sales comunes a excluir
    common_smiles = {
        'Cc1ccc(S(=O)(=O)[O-])cc1',         # p-toluensulfonato
        'O=S(=O)([O-])c1ccccc1',             # bencenosulfonato
        'Cc1ccc(S(=O)(=O)O)cc1',             # ácido p-toluensulfónico
        'O=C(O)C(F)(F)F',                    # trifluoroacetato
        'O=C(O)C(=O)O',                      # oxalato
        'O=S(=O)(O)c1ccccc1',               # ácido bencenosulfónico
        'O=C(O)c1ccccc1',                   # benzoato
    }

    fragments = list(Chem.GetMolFrags(mol, asMols=True))
    num_fragments = len(fragments)

    # Filtrar: solo válidos y no en lista de sales comunes
    fragments = [f for f in fragments if is_valid(f)]
    fragments = [f for f in fragments
                 if Chem.MolToSmiles(f) not in common_smiles]

    unique_smiles   = list(set(Chem.MolToSmiles(f) for f in fragments))
    unique_fragments = [Chem.MolFromSmiles(s) for s in unique_smiles]
    num_unique = len(unique_fragments)

    if num_unique == 0:
        return None, num_fragments, num_unique, "Sin fragmentos orgánicos válidos"

    if num_unique == 1:
        return unique_fragments[0], num_fragments, num_unique, None

    # Seleccionar el más grande
    largest = max(unique_fragments, key=lambda x: x.GetNumHeavyAtoms())
    total_atoms = sum(Descriptors.HeavyAtomCount(f) for f in
                      [Chem.MolFromSmiles(s) for s in unique_smiles])
    if Descriptors.HeavyAtomCount(largest) < 0.6 * total_atoms:
        return None, num_fragments, num_unique, "Fragmento mayor < 60% del total de átomos"

    return largest, num_fragments, num_unique, None


# ── Demostración con ejemplos ────────────────────────────────────────────────
ejemplos = {
    'Sal simple':           '[Na+].[O-]C(=O)c1ccccc1NC(=O)c1ccc(F)cc1',
    'Dos activos':          'c1ccc(NC(=O)c2ccccc2)cc1.c1ccc(NC(=O)c2ccccc2)cc1',
    'Metal + orgánico':     '[Pt+2].[NH3].c1ccc(NC(=O)c2ccccc2)cc1',
    'SMILES limpio':        'CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1',
}

print("DEMOSTRACIÓN DE select_largest_organic_component()")
print("=" * 65)
for descripcion, smi in ejemplos.items():
    mol, error = read_smiles(smi)
    if mol:
        frag, n_frags, n_uniq, err = select_largest_organic_component(mol)
        resultado = Chem.MolToSmiles(frag)[:50] if frag else f'None ({err})'
        print(f"  [{descripcion}]")
        print(f"    Entrada:     {smi[:55]}")
        print(f"    Fragmentos:  {n_frags} total, {n_uniq} únicos válidos")
        print(f"    Salida:      {resultado}")
    print()


---
## 6. Estandarización con ChEMBL Structure Pipeline

Después de eliminar sales, estandarizamos la representación SMILES usando el  
**protocolo oficial de ChEMBL**. Esto garantiza:
- Neutralización de cargas cuando corresponde
- Eliminación de estereoquímica cuando es ambigua  
- SMILES canónico reproducible y consistente con ChEMBL


In [ ]:
# ── Función de estandarización (del notebook de docking) ────────────────────
def chembl_standardizer(mol):
    """
    Estandariza una molécula usando el protocolo oficial de ChEMBL Structure Pipeline.
    
    El protocolo incluye:
    - Normalización de cargas (neutralización cuando es apropiado)
    - Eliminación de fragmentos inorgánicos residuales
    - Estandarización de representación aromática
    - Corrección de errores de valencia
    
    Parámetros
    ----------
    mol : rdkit.Chem.Mol — molécula a estandarizar (ya sin sales)

    Retorna
    -------
    smiles : str o None — SMILES estandarizado
    error  : str o None — descripción del error si falla
    """
    try:
        mol_std = standardize_mol(mol)
        smiles  = Chem.MolToSmiles(mol_std)
        return smiles, None
    except Exception as e:
        return None, f"Error de estandarización: {str(e)[:60]}"


# ── Demostración: antes vs después ──────────────────────────────────────────
demos_std = {
    'Carga explícita':      'CC(=O)[O-].[Na+]',           # acetato de sodio → ácido acético
    'SMILES no canónico':   'O=C(NC1=CC=CC=C1)C',          # debería canonicar igual
    'Tautómero':            'Oc1ccncc1',                    # 4-hidroxipiridina / 4-piridinona
    'Erlotinib SMILES alt': 'C#Cc1cccc(Nc2ncnc3cc(OCC)c(OCC)cc23)c1',
}

print("DEMOSTRACIÓN DE chembl_standardizer()")
print("=" * 65)
for desc, smi in demos_std.items():
    mol, err = read_smiles(smi)
    if mol:
        frag, *_ , err_frag = select_largest_organic_component(mol)
        if frag:
            smi_std, err_std = chembl_standardizer(frag)
            smi_before = Chem.MolToSmiles(frag)
            print(f"  [{desc}]")
            print(f"    Antes:  {smi_before[:60]}")
            print(f"    Después:{smi_std[:60] if smi_std else f'None — {err_std}'}")
            cambia = '→ cambió' if smi_before != smi_std else '→ sin cambio'
            print(f"    {cambia}")
    print()


---
## 7. Pipeline completo: `process_smiles()`

Las funciones anteriores se combinan en un único pipeline que procesa cualquier SMILES
de ChEMBL en tres pasos secuenciales.


In [ ]:
# ── Pipeline completo (del notebook de docking, con mejoras) ─────────────────
def process_smiles(smiles):
    """
    Procesa un SMILES según el flujo de curación estándar:
    
    Paso 1: read_smiles()                  → valida que RDKit pueda leerlo
    Paso 2: select_largest_organic_component() → elimina sales y fragmentos
    Paso 3: chembl_standardizer()          → estandariza con protocolo ChEMBL
    
    Parámetros
    ----------
    smiles : str — SMILES a procesar (puede ser crudo de ChEMBL)

    Retorna
    -------
    new_smiles : str o None — SMILES curado, o None si falla en cualquier paso
    info       : dict — metadatos del proceso (num_frags, error, paso_fallo)
    """
    # Paso 1: Validar SMILES
    mol, error = read_smiles(smiles)
    if error:
        return None, {'paso': 1, 'error': error,
                      'num_fragments': None, 'num_unique_fragments': None}

    # Paso 2: Seleccionar fragmento principal
    largest, num_frags, num_uniq, error = select_largest_organic_component(mol)
    if error:
        return None, {'paso': 2, 'error': error,
                      'num_fragments': num_frags, 'num_unique_fragments': num_uniq}

    # Paso 3: Estandarizar
    final_smiles, error = chembl_standardizer(largest)
    if error:
        return None, {'paso': 3, 'error': error,
                      'num_fragments': num_frags, 'num_unique_fragments': num_uniq}

    return final_smiles, {
        'paso': None, 'error': None,
        'num_fragments': num_frags, 'num_unique_fragments': num_uniq
    }


# ── Diagrama del pipeline ────────────────────────────────────────────────────
print("PIPELINE DE CURACIÓN ESTRUCTURAL")
print("=" * 55)
print()
print("  SMILES crudo (ChEMBL)")
print("       │")
print("       ▼")
print("  ┌─────────────────────────────────────────┐")
print("  │  Paso 1: read_smiles()                  │")
print("  │  → ¿Puede RDKit leer el SMILES?         │")
print("  │  → Rechaza: SMILES vacíos o malformados │")
print("  └────────────────────┬────────────────────┘")
print("                       │ mol válido")
print("                       ▼")
print("  ┌─────────────────────────────────────────┐")
print("  │  Paso 2: select_largest_organic_component│")
print("  │  → Separa fragmentos (Chem.GetMolFrags) │")
print("  │  → Excluye sales comunes                │")
print("  │  → Verifica: anillo + aceptor H         │")
print("  │  → Fragmento mayor ≥ 60% átomos totales │")
print("  └────────────────────┬────────────────────┘")
print("                       │ fragmento principal")
print("                       ▼")
print("  ┌─────────────────────────────────────────┐")
print("  │  Paso 3: chembl_standardizer()          │")
print("  │  → Protocolo oficial ChEMBL             │")
print("  │  → Normaliza cargas, tautómeros         │")
print("  │  → SMILES canónico reproducible         │")
print("  └────────────────────┬────────────────────┘")
print("                       │")
print("                       ▼")
print("  SMILES curado + metadatos de proceso")


In [ ]:
# ── Aplicar process_smiles() a todo el dataset ───────────────────────────────
from tqdm.auto import tqdm

print(f"Procesando {len(df_raw)} SMILES...")
resultados = [process_smiles(smi) for smi in tqdm(df_raw['canonical_smiles'])]

df_raw['std_smiles']         = [r[0] for r in resultados]
df_raw['cur_paso_fallo']     = [r[1]['paso'] for r in resultados]
df_raw['cur_error']          = [r[1]['error'] for r in resultados]
df_raw['cur_num_fragmentos'] = [r[1]['num_fragments'] for r in resultados]

# ── Reporte de curación estructural ─────────────────────────────────────────
exitosos   = df_raw['std_smiles'].notna().sum()
fallidos   = df_raw['std_smiles'].isna().sum()

print()
print("RESULTADO DE LA CURACIÓN ESTRUCTURAL")
print("=" * 45)
print(f"  ✅ Procesados correctamente: {exitosos}")
print(f"  ❌ Descartados:              {fallidos}")
print(f"  Tasa de retención:          {exitosos/len(df_raw)*100:.1f}%")
print()
print("Desglose de fallos por paso:")
for paso in [1, 2, 3]:
    n = (df_raw['cur_paso_fallo'] == paso).sum()
    label = {1:'SMILES inválido',
             2:'Sin fragmento válido',
             3:'Error de estandarización'}[paso]
    print(f"  Paso {paso} ({label}): {n}")


In [ ]:
# ── Ver los errores más frecuentes ──────────────────────────────────────────
if fallidos > 0:
    errores = df_raw[df_raw['std_smiles'].isna()]['cur_error'].value_counts()
    print("ERRORES MÁS FRECUENTES")
    print("=" * 50)
    for err, n in errores.head(8).items():
        print(f"  {n:>4}  {err}")


---
## 8. Curación de la actividad biológica

La estructura química ya está curada. Ahora curamos los **valores numéricos de actividad**:
duplicados, outliers, homogeneización entre tipos de actividad.


In [ ]:
# ── Trabajar solo con moléculas que pasaron la curación estructural ──────────
df_ok = df_raw[df_raw['std_smiles'].notna()].copy()

print(f"Dataset para curación de actividad: {len(df_ok)} registros")
print()

# ── 1. Detectar duplicados exactos (mismo SMILES curado + mismo tipo) ────────
dup_mask = df_ok.duplicated(subset=['std_smiles', 'standard_type'], keep=False)
print(f"Registros duplicados (mismo SMILES + tipo): {dup_mask.sum()}")
print()
if dup_mask.sum() > 0:
    ejemplo_dup = df_ok[dup_mask].groupby(['std_smiles','standard_type']).size()
    print("Grupos de duplicados (top 5):")
    print(ejemplo_dup.sort_values(ascending=False).head(5))


In [ ]:
# ── 2. Estrategia para manejar duplicados ────────────────────────────────────
# Cuando hay múltiples medidas para el mismo compuesto y tipo de actividad:
#   Opción A: Mediana (robusta a outliers) ← recomendada para drug discovery
#   Opción B: Media
#   Opción C: Mínimo (más conservador, la medida "más potente")

# Agrupar por SMILES curado + tipo de actividad → mediana
df_dedup = (df_ok.groupby(['std_smiles', 'standard_type'], as_index=False)
            .agg(
                molecule_chembl_id=('molecule_chembl_id', 'first'),
                valor_nM_mediana=('valor_nM', 'median'),
                valor_nM_n=('valor_nM', 'count'),
                valor_nM_std=('valor_nM', 'std'),
                pActividad_mediana=('pActividad', 'median'),
            ))

print("DESPUÉS DE DEDUPLICACIÓN (mediana)")
print("=" * 45)
print(f"  Antes:  {len(df_ok)} registros")
print(f"  Después:{len(df_dedup)} registros")
print(f"  Reducción: {(1 - len(df_dedup)/len(df_ok))*100:.1f}%")
print()
print(df_dedup.head(5).to_string(index=False))


In [ ]:
# ── 3. Detectar y tratar outliers ────────────────────────────────────────────
# Criterio: valores extremos de pActividad (< 3 o > 12) son sospechosos

LIMITE_INFERIOR_P = 3.0   # IC50 > 1 mM — difícilmente relevante
LIMITE_SUPERIOR_P = 12.0  # IC50 < 1 pM — inusual, probable error

print("ANÁLISIS DE OUTLIERS EN pActividad")
print("=" * 50)

outliers_bajos = df_dedup[df_dedup['pActividad_mediana'] < LIMITE_INFERIOR_P]
outliers_altos = df_dedup[df_dedup['pActividad_mediana'] > LIMITE_SUPERIOR_P]

print(f"  pActiv < {LIMITE_INFERIOR_P} (IC50 > 1mM):  {len(outliers_bajos)} registros")
print(f"  pActiv > {LIMITE_SUPERIOR_P} (IC50 < 1pM):  {len(outliers_altos)} registros")

if len(outliers_altos) > 0:
    print()
    print("Outliers con pActividad extremadamente alta:")
    print(outliers_altos[['std_smiles','standard_type',
                           'pActividad_mediana','valor_nM_n']].head(5).to_string(index=False))
    print("⚠️  Verificar en el artículo original antes de descartar")


In [ ]:
# ── 4. Filtrar outliers y aplicar umbral de calidad ─────────────────────────
df_curado_act = df_dedup[
    (df_dedup['pActividad_mediana'] >= LIMITE_INFERIOR_P) &
    (df_dedup['pActividad_mediana'] <= LIMITE_SUPERIOR_P)
].copy()

print(f"Dataset tras filtro de outliers: {len(df_curado_act)} registros")
print()

# ── 5. Incertidumbre de la medida: desviación estándar ─────────────────────
# Compuestos medidos varias veces: verificar consistencia
df_multiples = df_curado_act[df_curado_act['valor_nM_n'] > 1].copy()
if len(df_multiples) > 0:
    df_multiples['cv'] = (df_multiples['valor_nM_std'] /
                          df_multiples['valor_nM_mediana'] * 100)
    alta_variabilidad = df_multiples[df_multiples['cv'] > 100]
    print(f"Compuestos con >1 medición: {len(df_multiples)}")
    print(f"Alta variabilidad (CV>100%): {len(alta_variabilidad)}")
    print()
    print("💡 Un CV>100% sugiere mediciones inconsistentes entre laboratorios.")
    print("   En la semana 4, esto influirá en la incertidumbre del modelo.")


---
## 9. Filtros adicionales: drug-likeness y PAINS

Más allá de la curación de datos, aplicamos filtros farmacológicos  
para asegurar que las moléculas son candidatas razonables.


In [ ]:
# ── Calcular descriptores fisicoquímicos para todos los compuestos ───────────
def calcular_descriptores(smiles):
    """Calcula descriptores de Lipinski y otros relevantes desde un SMILES curado."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {k: None for k in ['MW','LogP','HBD','HBA','TPSA','RotBonds',
                                   'Rings','HeavyAtoms','QED']}
    from rdkit.Chem import QED
    return {
        'MW':         round(Descriptors.MolWt(mol), 2),
        'LogP':       round(Descriptors.MolLogP(mol), 2),
        'HBD':        Descriptors.NumHDonors(mol),
        'HBA':        Descriptors.NumHAcceptors(mol),
        'TPSA':       round(Descriptors.TPSA(mol), 2),
        'RotBonds':   Descriptors.NumRotatableBonds(mol),
        'Rings':      Descriptors.RingCount(mol),
        'HeavyAtoms': mol.GetNumHeavyAtoms(),
        'QED':        round(QED.qed(mol), 3),
    }

print("Calculando descriptores fisicoquímicos...")
descriptores = [calcular_descriptores(smi) for smi in df_curado_act['std_smiles']]
df_desc = pd.DataFrame(descriptores)
df_curado_act = pd.concat([df_curado_act.reset_index(drop=True), df_desc], axis=1)

print(f"✅ Descriptores calculados para {len(df_curado_act)} compuestos")
print()
print("Estadísticas de descriptores:")
print(df_curado_act[['MW','LogP','HBD','HBA','TPSA','RotBonds','QED']].describe().round(2))


In [ ]:
# ── Aplicar Regla de los Cinco de Lipinski ──────────────────────────────────
def cumple_lipinski(row):
    """Retorna True si la molécula cumple la Regla de los 5 de Lipinski."""
    violations = 0
    if row.get('MW',  9999) > 500: violations += 1
    if row.get('LogP', 999) > 5:   violations += 1
    if row.get('HBD',  999) > 5:   violations += 1
    if row.get('HBA',  999) > 10:  violations += 1
    return violations <= 1   # tolerar 1 violación (regla estándar)

df_curado_act['lipinski_ok'] = df_curado_act.apply(cumple_lipinski, axis=1)

n_lip_ok  = df_curado_act['lipinski_ok'].sum()
n_lip_no  = (~df_curado_act['lipinski_ok']).sum()

print("FILTRO DE LIPINSKI (≤1 violación permitida)")
print("=" * 45)
print(f"  Cumplen Lipinski:     {n_lip_ok} ({n_lip_ok/len(df_curado_act)*100:.1f}%)")
print(f"  No cumplen Lipinski:  {n_lip_no} ({n_lip_no/len(df_curado_act)*100:.1f}%)")
print()
print("💡 En drug discovery se suele conservar todos los compuestos")
print("   y usar Lipinski como señal, no como filtro duro.")
print("   Para inhibidores de PPI o antibióticos, el umbral puede ser mayor.")


In [ ]:
# ── Filtro PAINS (Pan-Assay Interference Compounds) ─────────────────────────
# Los PAINS son compuestos que dan falsos positivos en ensayos de HTS
# por mecanismos no específicos (agregación, reactividad, fluorescencia)

params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog_pains = FilterCatalog.FilterCatalog(params)

def es_pains(smiles):
    """Retorna True si la molécula es un PAINS."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    return catalog_pains.HasMatch(mol)

def razon_pains(smiles):
    """Retorna la categoría PAINS si aplica."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    entry = catalog_pains.GetFirstMatch(mol)
    if entry:
        return entry.GetDescription()
    return None

df_curado_act['es_pains']   = [es_pains(s) for s in df_curado_act['std_smiles']]
df_curado_act['razon_pains'] = [razon_pains(s) for s in df_curado_act['std_smiles']]

n_pains = df_curado_act['es_pains'].sum()
print("FILTRO PAINS")
print("=" * 45)
print(f"  PAINS detectados: {n_pains} ({n_pains/len(df_curado_act)*100:.1f}%)")
print()
if n_pains > 0:
    pains_counts = df_curado_act[df_curado_act['es_pains']]['razon_pains'].value_counts()
    print("Categorías PAINS más frecuentes:")
    for cat, n in pains_counts.head(5).items():
        print(f"  {n:>4}  {cat}")
    print()
    print("⚠️  Los PAINS se marcan pero no se eliminan automáticamente.")
    print("   Deben revisarse caso a caso con el químico medicinal.")


In [ ]:
# ── Visualizar distribución de descriptores por grupo ───────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

descriptores_plot = [
    ('MW',       'Peso molecular (Da)',  500,  '#4a90d9'),
    ('LogP',     'LogP (lipofilia)',     5,    '#e74c3c'),
    ('HBD',      'Donadores H-bond',    5,    '#27ae60'),
    ('HBA',      'Aceptores H-bond',    10,   '#f39c12'),
    ('TPSA',     'TPSA (Å²)',           140,  '#9b59b6'),
    ('QED',      'QED (drug-likeness)', None, '#1abc9c'),
]

for ax, (col, label, limite, color) in zip(axes, descriptores_plot):
    datos = df_curado_act[col].dropna()
    ax.hist(datos, bins=35, color=color, alpha=0.75,
            edgecolor='white', linewidth=0.4)
    if limite is not None:
        ax.axvline(limite, color='black', linestyle='--',
                   linewidth=1.5, label=f'Límite Lipinski: {limite}')
        ax.legend(fontsize=8)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Frecuencia', fontsize=10)
    mediana = datos.median()
    ax.set_title(f'{label}\nMediana: {mediana:.1f}', fontsize=10)

plt.suptitle(f'Descriptores fisicoquímicos — {TARGET_NAME} (curado)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('descriptores_curados.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 10. Clasificación activo / inactivo

Para construir modelos de clasificación (Semana 4), necesitamos  
asignar una etiqueta binaria a cada compuesto. La elección del umbral es una **decisión científica**, no arbitraria.


In [ ]:
# ── Discusión de umbrales ────────────────────────────────────────────────────
print("CRITERIOS PARA ELEGIR EL UMBRAL DE CLASIFICACIÓN")
print("=" * 60)
print()
print("  pActiv  │  Valor absoluto  │  Interpretación biológica")
print("  " + "-"*60)
criterios = [
    (5,  '10 µM',  'Límite estándar — activo débil pero detectable'),
    (6,  '1 µM',   'Activo moderado — umbral común en HTS primario'),
    (7,  '100 nM', 'Activo — umbral exigente, usado en lead optimization'),
    (8,  '10 nM',  'Muy activo — candidato a lead compound'),
    (9,  '1 nM',   'Extremadamente potente — candidato clínico'),
]
for p, val, interp in criterios:
    print(f"  pAct={p}  │  {val:<14}  │  {interp}")
print()
print("💡 Para EGFR con inhibidores aprobados (erlotinib pIC50≈9),")
print("   usaremos pActividad ≥ 6 (IC50 ≤ 1µM) como 'Activo'.")
print("   Esto es debatible — en tu proyecto, justifica tu elección.")


In [ ]:
# ── Aplicar umbral y generar etiqueta ───────────────────────────────────────
UMBRAL_PACTIVIDAD = 6.0    # ← ajusta según tu target y objetivo

df_curado_act['activo'] = (df_curado_act['pActividad_mediana'] >= UMBRAL_PACTIVIDAD).astype(int)

n_activos   = df_curado_act['activo'].sum()
n_inactivos = (df_curado_act['activo'] == 0).sum()
ratio = n_activos / n_inactivos if n_inactivos > 0 else float('inf')

print(f"CLASIFICACIÓN ACTIVO/INACTIVO (umbral pAct={UMBRAL_PACTIVIDAD})")
print("=" * 50)
print(f"  Activos   (pAct ≥ {UMBRAL_PACTIVIDAD}): {n_activos:>5}  ({n_activos/len(df_curado_act)*100:.1f}%)")
print(f"  Inactivos (pAct <  {UMBRAL_PACTIVIDAD}): {n_inactivos:>5}  ({n_inactivos/len(df_curado_act)*100:.1f}%)")
print(f"  Ratio activos/inactivos: {ratio:.2f}")
print()
if ratio < 0.2 or ratio > 5:
    print("⚠️  Dataset desbalanceado. En la Semana 4 aplicaremos técnicas")
    print("   de balanceo (oversampling, undersampling, class_weight).")
else:
    print("✅ Balance razonable para entrenamiento de modelos.")


In [ ]:
# ── Visualización: distribución y umbral ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma con umbral
ax1 = axes[0]
activos_data   = df_curado_act[df_curado_act['activo']==1]['pActividad_mediana']
inactivos_data = df_curado_act[df_curado_act['activo']==0]['pActividad_mediana']
ax1.hist(inactivos_data, bins=30, color='#e74c3c', alpha=0.7, label='Inactivo')
ax1.hist(activos_data,   bins=30, color='#27ae60', alpha=0.7, label='Activo')
ax1.axvline(UMBRAL_PACTIVIDAD, color='black', linestyle='--',
            linewidth=2, label=f'Umbral = {UMBRAL_PACTIVIDAD}')
ax1.set_xlabel('pActividad', fontsize=11)
ax1.set_ylabel('Frecuencia', fontsize=11)
ax1.set_title(f'Clasificación activo/inactivo\n{TARGET_NAME}', fontsize=11)
ax1.legend(fontsize=10)

# Pie chart
ax2 = axes[1]
sizes  = [n_activos, n_inactivos]
labels = [f'Activos\n({n_activos})', f'Inactivos\n({n_inactivos})']
colors = ['#27ae60', '#e74c3c']
wedges, texts, autotexts = ax2.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 11}
)
ax2.set_title(f'Balance de clases\n(umbral pAct={UMBRAL_PACTIVIDAD})', fontsize=11)

plt.tight_layout()
plt.savefig('clasificacion_activo_inactivo.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Zona gris: compuestos en el límite ──────────────────────────────────────
# Compuestos muy cercanos al umbral son ambiguos — mejor excluirlos del entrenamiento

MARGEN_GRIS = 0.5  # ± 0.5 unidades de pActividad alrededor del umbral

zona_gris = df_curado_act[
    (df_curado_act['pActividad_mediana'] >= UMBRAL_PACTIVIDAD - MARGEN_GRIS) &
    (df_curado_act['pActividad_mediana'] <= UMBRAL_PACTIVIDAD + MARGEN_GRIS)
]
print(f"ZONA GRIS (pAct entre {UMBRAL_PACTIVIDAD-MARGEN_GRIS} y {UMBRAL_PACTIVIDAD+MARGEN_GRIS})")
print(f"  Compuestos en zona gris: {len(zona_gris)}")
print()
print("💡 En la Semana 4 puedes excluir estos compuestos del entrenamiento")
print("   para que el modelo aprenda de los ejemplos más claros.")
print()
print("   Estrategia A: Excluir zona gris completamente")
print("   Estrategia B: Incluir con peso reducido (sample_weight)")
print("   Estrategia C: Usar regresión en lugar de clasificación")


---
## 11. Reporte final de curación

Un buen notebook de curación siempre termina con un **reporte de trazabilidad**:
cuántos compuestos se perdieron en cada paso y por qué.


In [ ]:
# ── Construir el reporte de curación ────────────────────────────────────────
n_original       = len(df_raw)
n_tras_smiles    = df_raw['std_smiles'].notna().sum()
n_tras_actividad = len(df_curado_act)
n_dataset_final  = len(df_curado_act)

print("REPORTE DE CURACIÓN — " + TARGET_NAME.upper())
print("=" * 60)
print()
print(f"  {'Paso':<40} {'n':>6}  {'%':>6}  {'Δ':>6}")
print("  " + "-"*58)

pasos = [
    ("Dataset raw descargado",                     n_original,       100.0,             0),
    ("Tras curación estructural (process_smiles)",  n_tras_smiles,
     n_tras_smiles/n_original*100,   n_original - n_tras_smiles),
    ("Tras deduplicación y outliers",               n_tras_actividad,
     n_tras_actividad/n_original*100, n_tras_smiles - n_tras_actividad),
]

for label, n, pct, delta in pasos:
    delta_str = f'-{delta}' if delta > 0 else '0'
    print(f"  {label:<40} {n:>6}  {pct:>5.1f}%  {delta_str:>6}")

print("  " + "-"*58)
print()
print(f"  Activos en dataset final:  {n_activos}")
print(f"  Inactivos en dataset final:{n_inactivos}")
print(f"  PAINS marcados:            {df_curado_act['es_pains'].sum()}")
print()

# Resumen visual de retención
barra_retencion = int(n_dataset_final / n_original * 40)
barra_perdida   = 40 - barra_retencion
print("  Retención:")
print(f"  [{'█' * barra_retencion}{'░' * barra_perdida}] {n_dataset_final/n_original*100:.1f}%")


In [ ]:
# ── Visualización del funnel de curación ─────────────────────────────────────
etapas = ['Raw\ndescargado', 'Curación\nestructural',
           'Dedup +\noutliers', 'Clasificados\n(final)']
valores = [n_original, n_tras_smiles, n_tras_actividad, n_dataset_final]

fig, ax = plt.subplots(figsize=(9, 4))
colores = ['#4a90d9','#27ae60','#f39c12','#9b59b6']
bars = ax.bar(etapas, valores, color=colores, alpha=0.8,
              edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val}', ha='center', va='bottom', fontsize=11, fontweight='bold')

for i in range(len(valores)-1):
    perdidos = valores[i] - valores[i+1]
    pct = perdidos / valores[i] * 100 if valores[i] > 0 else 0
    ax.annotate(f'−{perdidos}\n({pct:.1f}%)',
                xy=((i + i+1)/2, (valores[i] + valores[i+1])/2),
                fontsize=9, color='#e74c3c', ha='center', style='italic')

ax.set_ylabel('Número de compuestos', fontsize=11)
ax.set_title(f'Funnel de curación — {TARGET_NAME}', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(valores) * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('funnel_curacion.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 12. Guardar el dataset curado

El dataset curado es la entrada para la **Semana 4** (modelos QSAR y Random Forest).


In [ ]:
# ── Seleccionar columnas finales ─────────────────────────────────────────────
columnas_finales = [
    'molecule_chembl_id',    # identificador ChEMBL
    'std_smiles',            # SMILES curado y estandarizado
    'standard_type',         # tipo de actividad (IC50, Ki, ...)
    'valor_nM_mediana',      # valor en nM (mediana de réplicas)
    'valor_nM_n',            # número de mediciones (réplicas)
    'pActividad_mediana',    # pActividad = -log10(valor en M)
    'activo',                # 1 = activo, 0 = inactivo
    'MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds', 'QED',  # descriptores
    'lipinski_ok',           # cumple Lipinski
    'es_pains',              # es PAINS (interferente)
    'razon_pains',           # categoría PAINS si aplica
]
columnas_disponibles = [c for c in columnas_finales if c in df_curado_act.columns]
df_final = df_curado_act[columnas_disponibles].copy()

# Renombrar para claridad
df_final = df_final.rename(columns={
    'valor_nM_mediana':   'IC50_nM',
    'valor_nM_n':         'n_mediciones',
    'pActividad_mediana': 'pActividad',
})

print("DATASET CURADO — VISTA PREVIA")
print(f"Forma: {df_final.shape[0]} compuestos × {df_final.shape[1]} columnas")
print()
print(df_final.head(5).to_string(index=False))


In [ ]:
# ── Guardar CSV ──────────────────────────────────────────────────────────────
nombre_target_limpio = TARGET_NAME.replace(' ', '_').replace('/', '_').lower()
nombre_archivo = f'dataset_{nombre_target_limpio}_curado.csv'

df_final.to_csv(nombre_archivo, index=False)

print(f"✅ Dataset guardado: {nombre_archivo}")
print()
print("Columnas en el archivo:")
for col in df_final.columns:
    tipo  = df_final[col].dtype
    n_ok  = df_final[col].notna().sum()
    print(f"  {col:<25} {str(tipo):<12} {n_ok}/{len(df_final)} no nulos")
print()
print("Este archivo es la entrada del NB-ML-01 (Semana 4: modelos QSAR).")


---
## ✅ Resumen del notebook

| Paso | Función/herramienta | Qué hace |
|------|---------------------|----------|
| **1. Descarga** | `chembl_mols()` | Datos raw con filtro inicial de MW y pActividad |
| **2. Diagnóstico** | pandas + matplotlib | Detectar unidades, duplicados, SMILES problemáticos |
| **3. Unidades** | `convertir_a_nm()` | Homogeneizar a nM; calcular pActividad |
| **4. Validación** | `read_smiles()` | Verificar que RDKit lee el SMILES |
| **5. Sales** | `select_largest_organic_component()` | Eliminar contraiones, excluir sales comunes |
| **6. Estandarización** | `chembl_standardizer()` | Protocolo oficial ChEMBL Structure Pipeline |
| **7. Pipeline** | `process_smiles()` | Combina pasos 4–6 en una sola llamada |
| **8. Actividad** | pandas groupby | Mediana de réplicas, detección de outliers |
| **9. Filtros** | Lipinski + PAINS (RDKit) | Drug-likeness y compuestos interferentes |
| **10. Clasificación** | umbral pActividad | Etiqueta binaria activo/inactivo |
| **11. Reporte** | funnel plot | Trazabilidad de cuánto se perdió y por qué |
| **12. Guardar** | pandas to_csv | CSV listo para Semana 4 |

## 📅 Próximo notebook: NB-DATA-03 — Features y espacio químico

Con el dataset curado, calcularás descriptores moleculares,  
fingerprints de Morgan y visualizarás el espacio químico con PCA y t-SNE.

---
*NB-DATA-02 · Ciencia de Datos en Descubrimiento de Fármacos · UNAL 2026*  
*Funciones `process_smiles`, `select_largest_organic_component`, `chembl_standardizer`:  
basadas en el protocolo del notebook de docking del curso.*
